# Model Contribution with BaseModel as Parent

In [ ]:
"""3D CNN model for Alzheimer's disease detection from structural MRI.

Reference:
    Liu et al. "On the design of convolutional neural networks for automatic
    detection of Alzheimer's disease." ML4H @ NeurIPS 2019.
    https://arxiv.org/abs/1911.03740

    Liu et al. "Generalizable deep learning model for early Alzheimer's
    disease detection from structural MRIs." Scientific Reports, 2022.
    https://www.nature.com/articles/s41598-022-20674-x
"""

from typing import Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from pyhealth.datasets import SampleDataset
from pyhealth.models import BaseModel


class AlzheimerCNN(BaseModel):
    def __init__(
        self,
        dataset: SampleDataset,
        init_channels: int = 32,
        classifier_hidden_dim: int = 128,
        dropout: float = 0.5,
        **kwargs,
    ) -> None:
        super(AlzheimerCNN, self).__init__(dataset=dataset)

        self.init_channels = init_channels
        self.classifier_hidden_dim = classifier_hidden_dim

        # ── Block 1: Conv2d(1 → C) → InstanceNorm → LeakyReLU → MaxPool ──
        self.block1 = nn.Sequential(
            nn.Conv2d(1, init_channels, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 2: Conv2d(C → 2C) → InstanceNorm → LeakyReLU → MaxPool ─
        self.block2 = nn.Sequential(
            nn.Conv2d(init_channels, init_channels * 2, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 3: Conv2d(2C → 4C) → InstanceNorm → LeakyReLU → MaxPool ─
        self.block3 = nn.Sequential(
            nn.Conv2d(init_channels * 2, init_channels * 4, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 4),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 4: Conv2d(4C → 8C) → InstanceNorm → LeakyReLU → AdaptiveAvgPool ─
        self.block4 = nn.Sequential(
            nn.Conv2d(init_channels * 4, init_channels * 8, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 8),
            nn.LeakyReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # ── Classifier: FC(8C → hidden_dim) → LeakyReLU → Dropout → FC(hidden_dim → output) ─
        output_size = self.get_output_size()
        self.classifier = nn.Sequential(
            nn.Linear(init_channels * 8, classifier_hidden_dim),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(classifier_hidden_dim, output_size),
        )
    def forward(self, **kwargs) -> Dict[str, torch.Tensor]:
        # ── Extract inputs; self.device handles mps/cuda/cpu automatically ─
        x = kwargs[self.feature_keys[0]].to(self.device)
        labels = kwargs[self.label_keys[0]].to(self.device)

        # ── Convolutional backbone ────────────────────────────────────────
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        # ── Flatten: (B, 8C, 1, 1) → (B, 8C) ────────────────────────────
        emb = torch.flatten(x, 1)

        # ── Classifier head ───────────────────────────────────────────────
        logits = self.classifier(emb)

        # ── Loss + metrics via BaseModel helpers ──────────────────────────
        loss = self.get_loss_function()(logits, labels)
        y_prob = self.prepare_y_prob(logits)

        return {
            "loss": loss,
            "y_prob": y_prob,
            "y_true": labels,
            "logit": logits,
        }

# Data Pre-Processing

In [21]:
from datasets import load_dataset
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import torch

# ── Load from example 2d dataset
ds = load_dataset("Falah/Alzheimer_MRI")

# Transform the images and resize
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# Pyhealth Dataloader Class
#   dataset.input_schema                        -> feature key → type string
#   dataset.output_schema                       -> dict mapping label key → mode string
#   dataset.output_processors[label_key].size() ->  number of output classes (Non, Very Mild, Mild, Moderate)

class OutputProcessor:
    def __init__(self, num_classes: int):
        self.num_classes = num_classes

    def size(self) -> int:
        return self.num_classes


class AlzheimerMRIDataset(Dataset):
    """PyTorch Dataset that also exposes the schema attributes BaseModel needs.

    Args:
        hf_split: A HuggingFace dataset split (train or test).
        num_classes: Number of output classes. Default is 4.
    """

    #  Schema
    input_schema = {"image": "image"}
    output_schema = {"label": "multiclass"}

    def __init__(self, hf_split, num_classes: int = 4):
        self.data = hf_split
        self.num_classes = num_classes
        # BaseModel calls dataset.output_processors[label_key].size()
        self.output_processors = {
            "label": OutputProcessor(num_classes)
        }

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> dict:
        item = self.data[idx]
        return {
            "image": transform(item["image"]),  # (1, 128, 128) 
            "label": int(item["label"]),        # 0-3 integer for labeling
        }


# Data Instantiaion

In [22]:
# Instantiate the data
print("Wrapping train split...")
train_dataset = AlzheimerMRIDataset(ds["train"], num_classes=4)

print("Wrapping test split...")
test_dataset = AlzheimerMRIDataset(ds["test"], num_classes=4)

print(f"Train size: {len(train_dataset)}")
print(f"Test size:  {len(test_dataset)}")

# Load the data
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,  # 0 required on macOS MPS
)
val_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

Wrapping train split...
Wrapping test split...
Train size: 5120
Test size:  1280


# Model Instantiation and Training

In [ ]:
# Instantiate the Model
model = AlzheimerCNN(
    dataset=train_dataset,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
)

# Train the model
from pyhealth.trainer import Trainer

trainer = Trainer(
    model=model,
    metrics=["accuracy", "f1_macro"],
    device='mps',
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5},
)

AlzheimerCNN(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, c


Training:
Batch size: 32
Optimizer: <class 'torch.optim.adamw.AdamW'>
Optimizer params: {'lr': 0.0001, 'weight_decay': 1e-05}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x382951880>
Monitor: None
Monitor criterion: max
Epochs: 50
Patience: None



Epoch 0 / 50: 100%|██████████| 160/160 [00:46<00:00,  3.47it/s]

--- Train epoch-0, step-160 ---
loss: 1.0996



Evaluation: 100%|██████████| 40/40 [00:03<00:00, 11.42it/s]

...

--- Train epoch-4, step-800 ---
loss: 0.9584



Evaluation: 100%|██████████| 40/40 [00:03<00:00, 10.74it/s]

--- Eval epoch-4, step-800 ---
accuracy: 0.5695
f1_macro: 0.3004
loss: 0.9409




Epoch 5 / 50:  49%|████▉     | 78/160 [00:22<00:23,  3.48it/s]


KeyboardInterrupt: 

## The accuracy is quite high here and our loss decreased relative to the starting position. However, our f1 score suggests that there is a accuracy discrepancy between classifying certain classes. This is likely to happen if we have more results in one class than another, which impacts training/performance. 

# Ablation 3, CNN Transformer

In [24]:
class PatchEmbedding(nn.Module):
    """Converts a CNN feature map into a sequence of patch tokens.

    Splits a (B, C, H, W) feature map into non-overlapping patches of size
    patch_size x patch_size, projects each flattened patch to embed_dim via
    a linear layer, prepends a learnable [CLS] token, and adds learnable
    positional embeddings.

    Args:
        in_channels: Number of channels in the input feature map.
        patch_size: Height and width of each square patch.
        embed_dim: Dimensionality of the projected patch tokens.
        feature_map_size: Spatial height (and width) of the input feature map.
            Assumed to be square.
    """

    def __init__(
        self,
        in_channels: int,
        patch_size: int,
        embed_dim: int,
        feature_map_size: int,
    ) -> None:
        super().__init__()
        self.patch_size = patch_size
        num_patches = (feature_map_size // patch_size) ** 2

        # Each patch is flattened to (C * patch_size * patch_size) dims
        patch_dim = in_channels * patch_size * patch_size
        self.projection = nn.Linear(patch_dim, embed_dim)

        # Learnable [CLS] token — one per batch item
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Learnable positional embeddings — one per patch + one for [CLS]
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Tokenize a CNN feature map into a patch sequence with CLS token.

        Args:
            x: Feature map of shape (B, C, H, W).

        Returns:
            Token sequence of shape (B, num_patches + 1, embed_dim).
        """
        B, C, H, W = x.shape
        p = self.patch_size

        # (B, C, H, W) → (B, C, H//p, W//p, p, p)
        x = x.unfold(2, p, p).unfold(3, p, p)
        # → (B, C, num_patches, p*p)
        x = x.contiguous().view(B, C, -1, p * p)
        # → (B, num_patches, C, p*p)
        x = x.permute(0, 2, 1, 3)
        # → (B, num_patches, C*p*p)
        x = x.contiguous().view(B, -1, C * p * p)

        # Project each patch to embed_dim
        x = self.projection(x)                         # (B, num_patches, embed_dim)

        # Prepend [CLS] token and add positional embeddings
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)          # (B, num_patches + 1, embed_dim)
        x = x + self.pos_embedding

        return x


class AlzheimerCNNViT(BaseModel):
    """CNN + Vision Transformer hybrid for Alzheimer's disease classification.

    Combines a two-block CNN backbone for local feature extraction with a
    Transformer encoder for global context modelling over patch tokens. The
    CNN reduces the 128x128 input to a 32x32 feature map, which is then split
    into 4x4 patches and fed to the Transformer. Classification is performed
    on the output [CLS] token.

    Architecture::

        Input (B, 1, 128, 128)
            │
        block1: Conv2d → InstanceNorm2d → LeakyReLU → MaxPool  →  (B, C, 64, 64)
        block2: Conv2d → InstanceNorm2d → LeakyReLU → MaxPool  →  (B, 2C, 32, 32)
            │
        PatchEmbedding (patch_size=4) → (B, 65, embed_dim)
            │
        TransformerEncoder (num_transformer_layers)
            │
        [CLS] token → Linear → LeakyReLU → Dropout → Linear → logits

    Paper:
        Liu et al. "On the design of convolutional neural networks for
        automatic detection of Alzheimer's disease." ML4H @ NeurIPS 2019.
        https://arxiv.org/abs/1911.03740

    Args:
        dataset: The dataset to train the model. Must expose input_schema,
            output_schema, and output_processors to satisfy BaseModel.
        init_channels: Base number of CNN filters. Blocks use C and 2C
            channels. Default is 32.
        embed_dim: Dimensionality of Transformer patch token embeddings.
            Default is 128.
        num_heads: Number of attention heads in the Transformer encoder.
            Must evenly divide embed_dim. Default is 4.
        num_transformer_layers: Number of Transformer encoder layers.
            Default is 4.
        classifier_hidden_dim: Number of units in the hidden FC layer of
            the classifier head. Default is 128.
        dropout: Dropout probability applied in the Transformer and before
            the output linear layer. Default is 0.1.

    Examples:
        >>> model = AlzheimerCNNViT(dataset=train_dataset)
        >>> ret = model(**data_batch)
        >>> print(ret.keys())
        dict_keys(['loss', 'y_prob', 'y_true', 'logit'])
    """

    def __init__(
        self,
        dataset,
        init_channels: int = 32,
        embed_dim: int = 128,
        num_heads: int = 4,
        num_transformer_layers: int = 4,
        classifier_hidden_dim: int = 128,
        dropout: float = 0.1,
        **kwargs,
    ) -> None:
        super(AlzheimerCNNViT, self).__init__(dataset=dataset)

        self.init_channels = init_channels
        self.embed_dim = embed_dim

        # ── CNN backbone: local feature extraction ────────────────────────
        # Two pooling stages reduce 128×128 → 32×32
        self.block1 = nn.Sequential(
            nn.Conv2d(1, init_channels, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),          # 128 → 64
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(init_channels, init_channels * 2, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),          # 64 → 32
        )

        # ── Patch embedding: 32×32 feature map → 64 patch tokens + CLS ───
        # patch_size=4 gives (32/4)^2 = 64 patches
        self.patch_embed = PatchEmbedding(
            in_channels=init_channels * 2,
            patch_size=4,
            embed_dim=embed_dim,
            feature_map_size=32,
        )

        # ── Transformer encoder: global context over patch sequence ───────
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_transformer_layers,
        )

        # ── Classifier head: operates on the [CLS] token output ──────────
        output_size = self.get_output_size()
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, classifier_hidden_dim),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(classifier_hidden_dim, output_size),
        )

    def forward(self, **kwargs) -> Dict[str, torch.Tensor]:
        """Forward propagation.

        Args:
            **kwargs: Keys match the dataset's input_schema and output_schema.
                Expects an image tensor of shape (B, 1, H, W) and an integer
                label per sample.

        Returns:
            A dict with keys:
                - loss: scalar cross-entropy loss tensor.
                - y_prob: predicted probabilities, shape (B, num_classes).
                - y_true: ground-truth label tensor.
                - logit: raw logits, shape (B, num_classes).
        """
        # ── Extract inputs; self.device handles mps/cuda/cpu automatically ─
        x = kwargs[self.feature_keys[0]].to(self.device)
        labels = kwargs[self.label_keys[0]].to(self.device)

        # ── CNN local feature extraction ──────────────────────────────────
        x = self.block1(x)                     # (B, C,   64, 64)
        x = self.block2(x)                     # (B, 2C,  32, 32)

        # ── Patch tokenization ────────────────────────────────────────────
        x = self.patch_embed(x)                # (B, 65, embed_dim)

        # ── Transformer global context modelling ──────────────────────────
        x = self.transformer(x)                # (B, 65, embed_dim)

        # ── Extract [CLS] token (index 0) for classification ─────────────
        cls_output = x[:, 0, :]                # (B, embed_dim)

        # ── Classifier head ───────────────────────────────────────────────
        logits = self.classifier(cls_output)   # (B, num_classes)

        # ── Loss + metrics via BaseModel helpers ──────────────────────────
        loss = self.get_loss_function()(logits, labels)
        y_prob = self.prepare_y_prob(logits)

        return {
            "loss": loss,
            "y_prob": y_prob,
            "y_true": labels,
            "logit": logits,
        }

In [ ]:
# Instantiate the Model
model = AlzheimerCNNViT(
    dataset=train_dataset,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
)

# Train the model
from pyhealth.trainer import Trainer

trainer = Trainer(
    model=model,
    metrics=["accuracy", "f1_macro"],
    device='mps',
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5},
)

AlzheimerCNNViT(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (patch_embed): PatchEmbedding(
    (projection): Linear(in_features=1024, out_features=128, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_fe

Epoch 0 / 50: 100%|██████████| 160/160 [00:08<00:00, 19.69it/s]

--- Train epoch-0, step-160 ---
loss: 1.0924



Evaluation: 100%|██████████| 40/40 [00:00<00:00, 61.86it/s]

--- Eval epoch-0, step-160 ---
accuracy: 0.4953
f1_macro: 0.1656
loss: 1.0511



...

--- Eval epoch-48, step-7840 ---
accuracy: 0.8008
f1_macro: 0.6241
loss: 0.9327




Epoch 49 / 50: 100%|██████████| 160/160 [00:08<00:00, 18.74it/s]

--- Train epoch-49, step-8000 ---
loss: 0.0831



Evaluation: 100%|██████████| 40/40 [00:00<00:00, 63.66it/s]

--- Eval epoch-49, step-8000 ---
accuracy: 0.8484
f1_macro: 0.7259
loss: 0.6535


## The vision transformer underperformed our basemodel, likely due to the fact that we are using a 2-d dataset instead of a 3d one. We also are not using a huge dataset in this training mechanism. The vision transformer is theoretically better at grabbing attention at a larger global scale, which we do not have here. 

### On a second run, the ViT outperformed base, with a larger f1-macro and higher loss

# Ablation 1, Comparing Instance Norm to Group Norm Instead

In [26]:
class AlzheimerCNNNormVariant(BaseModel):
 
    def __init__(
        self,
        dataset,
        norm_type: str = "instance",
        num_groups: int = 8,
        init_channels: int = 32,
        classifier_hidden_dim: int = 128,
        dropout: float = 0.5,
        **kwargs,
    ) -> None:
        super(AlzheimerCNNNormVariant, self).__init__(dataset=dataset)

        if norm_type not in ("instance", "group", "layer"):
            raise ValueError(
                f"norm_type must be 'instance', 'group', or 'layer', got '{norm_type}'"
            )

        self.norm_type = norm_type
        self.num_groups = num_groups
        self.init_channels = init_channels

        def _get_norm(channels: int) -> nn.Module:
            """Build the appropriate normalization layer for a given channel count.

            Args:
                channels: Number of feature map channels to normalise.

            Returns:
                An nn.Module normalisation layer.
            """
            if norm_type == "instance":
                return nn.InstanceNorm2d(channels)
            elif norm_type == "group":
                # Ensure num_groups evenly divides channels
                groups = min(num_groups, channels)
                while channels % groups != 0:
                    groups //= 2
                return nn.GroupNorm(groups, channels)
            else:
                # layer norm: GroupNorm(1, C) is equivalent to
                # LayerNorm over the channel dimension
                return nn.GroupNorm(1, channels)

        # ── Block 1: Conv2d(1 → C) → Norm → LeakyReLU → MaxPool ──────────
        self.block1 = nn.Sequential(
            nn.Conv2d(1, init_channels, kernel_size=3, stride=1, padding=1),
            _get_norm(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 2: Conv2d(C → 2C) → Norm → LeakyReLU → MaxPool ─────────
        self.block2 = nn.Sequential(
            nn.Conv2d(init_channels, init_channels * 2, kernel_size=3, stride=1, padding=1),
            _get_norm(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 3: Conv2d(2C → 4C) → Norm → LeakyReLU → MaxPool ────────
        self.block3 = nn.Sequential(
            nn.Conv2d(init_channels * 2, init_channels * 4, kernel_size=3, stride=1, padding=1),
            _get_norm(init_channels * 4),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 4: Conv2d(4C → 8C) → Norm → LeakyReLU → AdaptiveAvgPool ─
        self.block4 = nn.Sequential(
            nn.Conv2d(init_channels * 4, init_channels * 8, kernel_size=3, stride=1, padding=1),
            _get_norm(init_channels * 8),
            nn.LeakyReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # ── Classifier: FC(8C → hidden) → LeakyReLU → Dropout → FC(hidden → output) ─
        output_size = self.get_output_size()
        self.classifier = nn.Sequential(
            nn.Linear(init_channels * 8, classifier_hidden_dim),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(classifier_hidden_dim, output_size),
        )

    def forward(self, **kwargs) -> Dict[str, torch.Tensor]:
        # Input and Device Management
        x = kwargs[self.feature_keys[0]].to(self.device)
        labels = kwargs[self.label_keys[0]].to(self.device)

        # Convolutions
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        # Flatten
        emb = torch.flatten(x, 1)

        logits = self.classifier(emb)

        # Metrics
        loss = self.get_loss_function()(logits, labels)
        y_prob = self.prepare_y_prob(logits)

        return {
            "loss": loss,
            "y_prob": y_prob,
            "y_true": labels,
            "logit": logits,
        }

In [ ]:
# Instantiate the Model
model = AlzheimerCNNNormVariant(
    dataset=train_dataset,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
    norm_type='group',
)

# Train the model
from pyhealth.trainer import Trainer

trainer = Trainer(
    model=model,
    metrics=["accuracy", "f1_macro"],
    device='mps',
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5},
)

AlzheimerCNNNormVariant(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))


Epoch 0 / 50: 100%|██████████| 160/160 [00:11<00:00, 14.32it/s]

--- Train epoch-0, step-160 ---
loss: 1.0777



Evaluation: 100%|██████████| 40/40 [00:00<00:00, 43.47it/s]

--- Eval epoch-0, step-160 ---
accuracy: 0.5148
f1_macro: 0.1984
loss: 1.0049



...

--- Eval epoch-48, step-7840 ---
accuracy: 0.5906
f1_macro: 0.3833
loss: 0.8123




Epoch 49 / 50: 100%|██████████| 160/160 [00:09<00:00, 17.75it/s]

--- Train epoch-49, step-8000 ---
loss: 0.7885



Evaluation: 100%|██████████| 40/40 [00:00<00:00, 43.08it/s]

--- Eval epoch-49, step-8000 ---
accuracy: 0.5867
f1_macro: 0.3909
loss: 0.8193


## Similar accurcy results, however, here we have significantly lowered the f1-macro which suggests this more venly trained on all classifications. 